In [0]:
# Silver layer: supplier_products (which supplier sells which product, at what price)
# Source: b2b_ecommerce.bronze.supplier_products (append-only, connector-managed
#         and already typed: int ids/qty/lead_time, decimal(10,2) price,
#         boolean is_preferred, DATE valid_from, timestamps)
# Grain:  one row per supplier_id + product_id (upsert - latest
#         updated_at wins, no history). Profiled 2026-11-11:
#         313 rows, 313 distinct (supplier_id, product_id) pairs, 313 distinct
#         supplier_sku, 0 nulls in every column, 0 orphans vs current
#         silver.suppliers and silver.products, price 2.15-393.63, qty 1-25,
#         lead_time 2-30, valid_from always before created_at.
# Approach: bronze is already clean (0 nulls, 0 padding, 0 orphans, all values
#           in range), so we only standardize supplier_sku (trim + uppercase)
#           and keep the typed columns as-is. AUTO CDC keyed by the composite
#           (supplier_id, product_id) MERGEs into b2b_ecommerce.silver
#           .supplier_products, which deduplicates identical re-sends and keeps
#           re-ingestion idempotent. Upsert (SCD Type 1): 33 of 313 rows have
#           updated_at > created_at, i.e. supplier-product prices ARE updated
#           at the source - a changed row re-synced into bronze overwrites the
#           existing key in place; only the latest state is kept, no history
#           (same choice as every other silver table).
from pyspark import pipelines as dp
from pyspark.sql import functions as F

BRONZE_SUPPLIER_PRODUCTS = "b2b_ecommerce.bronze.supplier_products"
# current suppliers and products live in the silver layer (same pipeline)
SILVER_SUPPLIERS = "suppliers"
SILVER_PRODUCTS = "products"


def clean_text(name, upper=False):
    """Trim a text column; a blank value ("" or " ") becomes NULL, not ""."""
    trimmed = F.trim(F.col(name))
    if upper:
        trimmed = F.upper(trimmed)
    return (
        F.when(trimmed != "", trimmed)
        .otherwise(F.lit(None).cast("string"))
        .alias(name)
    )


# ---------------------------------------------------------------------------
# 1) Cleaned stream of bronze.supplier_products (+ FK match flags vs current
#    silver.suppliers and silver.products rows)
# ---------------------------------------------------------------------------
@dp.temporary_view(name="supplier_products_clean_stream")
# Data quality validation (only checks that can actually catch bad rows).
# Drop-level: without the merge keys or the CDC sequence the row cannot be
# merged at all - quarantine it out of silver; the raw value stays in
# bronze for inspection.
@dp.expect_or_drop("supplier_id_not_null", "supplier_id IS NOT NULL")
@dp.expect_or_drop("product_id_not_null", "product_id IS NOT NULL")
@dp.expect_or_drop("updated_at_not_null", "updated_at IS NOT NULL")
# Referential integrity: every row must reference a CURRENT silver supplier
# AND a CURRENT silver product - a missing one fails the whole update instead
# of silently landing in silver (hard stop is intended, same policy as
# silver.customers / suppliers; this is a small bridge table, so an orphan
# means the source is broken and someone must intervene).
@dp.expect_or_fail("supplier_id_exists", "_supplier_id_match IS NOT NULL")
@dp.expect_or_fail("product_id_exists", "_product_id_match IS NOT NULL")
# Warn-level: the row is kept and the problem only recorded in the event
# log (same policy as silver.customers / suppliers value checks).
@dp.expect("supplier_sku_not_null", "supplier_sku IS NOT NULL")
@dp.expect("supplier_price_positive", "supplier_price IS NULL OR supplier_price > 0")
@dp.expect("min_order_qty_at_least_one", "min_order_qty IS NULL OR min_order_qty >= 1")
@dp.expect("lead_time_days_not_negative", "lead_time_days IS NULL OR lead_time_days >= 0")
def supplier_products_clean_stream():
    # bronze.supplier_products is append-only: a changed supplier-product
    # arrives as a new row. The AUTO CDC merge below turns it into an
    # in-place update of the same (supplier_id, product_id) key (upsert:
    # latest updated_at wins, no history), so identical re-sends collapse
    # and re-ingestion is idempotent.
    bronze = spark.readStream.table(BRONZE_SUPPLIER_PRODUCTS)

    # stream-static LEFT JOINs against CURRENT silver rows only: a supplier or
    # product that exists in bronze but was dropped from silver does not match
    # - the row becomes an orphan and the fail-level expectation stops the
    # update. An expectation is a per-row column expression, so each FK check
    # needs its match flag as a column.
    # silver.suppliers and silver.products are upserts (SCD Type 1): one
    # row per key, no version columns - every row is current
    suppliers = (
        spark.read.table(SILVER_SUPPLIERS)
        .select(F.col("supplier_id").alias("_supplier_id_match"))
        .distinct()
    )
    products = (
        spark.read.table(SILVER_PRODUCTS)
        .select(F.col("product_id").alias("_product_id_match"))
        .distinct()
    )

    return (
        bronze.join(suppliers, bronze["supplier_id"] == suppliers["_supplier_id_match"], "left")
        .join(products, bronze["product_id"] == products["_product_id_match"], "left")
        .select(
            F.col("supplier_id").cast("int").alias("supplier_id"),
            F.col("product_id").cast("int").alias("product_id"),
            F.col("supplier_price").cast("decimal(10,2)").alias("supplier_price"),
            # canonical uppercase SKU (same form as silver.products sku)
            clean_text("supplier_sku", upper=True),
            F.col("min_order_qty").cast("int").alias("min_order_qty"),
            F.col("lead_time_days").cast("int").alias("lead_time_days"),
            F.col("is_preferred").cast("boolean").alias("is_preferred"),
            F.col("valid_from").cast("date").alias("valid_from"),
            F.col("created_at").cast("timestamp").alias("created_at"),
            F.col("updated_at").cast("timestamp").alias("updated_at"),
            # internal FK flags, never land in silver
            F.col("_supplier_id_match"),
            F.col("_product_id_match"),
        )
    )


# ---------------------------------------------------------------------------
# 2) Silver target: b2b_ecommerce.silver.supplier_products
#    Rows are MERGEd by the composite (supplier_id, product_id) key
#    (upsert): identical re-sends update in place, a genuinely changed
#    row overwrites the existing key - only the latest state is kept,
#    no history.
# ---------------------------------------------------------------------------
dp.create_streaming_table(
    name="supplier_products",
    comment=(
        "Silver supplier_products: cleaned and standardized; one row per "
        "supplier_id + product_id (upsert merge - latest updated_at wins, "
        "no history)"
    ),
)

dp.create_auto_cdc_flow(
    target="supplier_products",
    source="supplier_products_clean_stream",
    keys=["supplier_id", "product_id"],
    sequence_by="updated_at",
    stored_as_scd_type=1,
    # internal FK flags must not land in the target table
    except_column_list=["_supplier_id_match", "_product_id_match"],
)